## 1. 安裝套件


In [ ]:
'''
Import the tools for installing ComfyUI.
匯入安裝 ComfyUI 所需工具。
'''
import subprocess
import sys
from pathlib import Path

'''
Use Kaggle's writable workspace.
使用 Kaggle 可寫入的工作目錄。
'''
ROOT = Path('/kaggle/working')

'''
Isolate the Z-Image ComfyUI installation from LTX-2.5 notebooks.
將 Z-Image 的 ComfyUI 與 LTX-2.5 Notebook 隔離，避免自訂節點衝突。
'''
COMFY = ROOT / 'ComfyUI-ZImageTurbo'
REPOS = [
    ('https://github.com/Comfy-Org/ComfyUI.git', COMFY),
    ('https://github.com/city96/ComfyUI-GGUF.git', COMFY / 'custom_nodes/ComfyUI-GGUF'),
    ('https://github.com/pollockjj/ComfyUI-MultiGPU.git', COMFY / 'custom_nodes/ComfyUI-MultiGPU'),
]

'''
Select the current verified stable ComfyUI release.
指定目前經確認的 ComfyUI 穩定版。
'''
COMFY_REF = 'v0.45.0'

'''
Record the previous source revisions to detect stale running servers.
記錄舊版原始碼版本，以偵測未重新啟動的服務。
'''
old_commits = {}

'''
Clone missing repositories and update only unmodified source trees.
安裝缺少的倉庫，僅更新未經本機修改的原始碼。
'''
for url, folder in REPOS:

    '''
    Reject an existing folder without Git metadata.
    避免覆寫來源不明的既有資料夾。
    '''
    if folder.exists() and not (folder / '.git').exists():

        '''
        Abort before changing arbitrary local files.
        在變更使用者檔案前停止。
        '''
        raise RuntimeError(f'不是 Git 倉庫：{folder}')

    '''
    Record the old commit for a previously installed component.
    記錄已安裝套件的舊版 commit。
    '''
    if (folder / '.git').exists():

        '''
        Read the current source revision.
        讀取目前的原始碼版本。
        '''
        old_commits[folder] = subprocess.check_output(['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True).strip()
    else:

        '''
        Download only missing source repositories.
        僅下載缺少的原始碼。
        '''
        command = ['git', 'clone', '-q', '--depth', '1']

        '''
        Install ComfyUI at the verified official release tag.
        安裝 ComfyUI 時指定官方穩定版標籤。
        '''
        if folder == COMFY:

            '''
            Select the exact official release.
            選擇確切的官方正式版本。
            '''
            command += ['--branch', COMFY_REF]

        '''
        Clone the repository into its original folder.
        Clone 到原有的專案目錄。
        '''
        subprocess.run(command + [url, str(folder)], check=True)

    '''
    MultiGPU's archived v2.6.4 source is no longer maintained.
    MultiGPU 已封存且停止維護，保留原有的最後版本。
    '''
    if folder.name == 'ComfyUI-MultiGPU':

        '''
        Do not replace archived integration with an unverified fork.
        不將封存的整合節點替換為未驗證的 Fork。
        '''
        print('MultiGPU v2.6.4 已停止維護，與新版 ComfyUI 的相容性仍須在 GPU 上驗證。')

    elif folder != COMFY:

        '''
        Update the GGUF custom node only when local changes are absent.
        GGUF 外掛沒有本機修改時才更新。
        '''

        '''
        Keep user modifications intact rather than force-resetting them.
        保留使用者修改，不進行強制重設。
        '''
        if not subprocess.check_output(['git', '-C', str(folder), 'status', '--porcelain'], text=True).strip():

            '''
            Apply only fast-forward updates from upstream.
            僅允許上游的 fast-forward 更新。
            '''
            subprocess.run(['git', '-C', str(folder), 'pull', '--ff-only', '-q'], check=True)
        else:

            '''
            Explain why the existing plugin was not updated.
            說明因本機修改而略過更新。
            '''
            print(f'略過已有修改的外掛：{folder.name}')

'''
Retrieve the official ComfyUI release even for an existing checkout.
現有 ComfyUI 倉庫也要取得指定的正式發行標籤。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)

'''
Check out the exact stable version instead of the moving development branch.
固定於穩定版本，不使用隨時變動的開發分支。
'''
subprocess.run(['git', '-C', str(COMFY), 'checkout', '--quiet', '--detach', COMFY_REF], check=True)

'''
Detect changes that require the existing server to restart.
偵測是否需要讓既有服務重啟。
'''
ZIMAGE_CODE_UPDATED = any(
    folder in old_commits and old_commits[folder] != subprocess.check_output(
        ['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for _, folder in REPOS
)

'''
Report all active upstream revisions.
顯示每個原始碼倉庫的實際版本。
'''
for _, folder in REPOS:

    '''
    Print the short commit for reproducible debugging.
    印出簡短 commit 以便重現與除錯。
    '''
    print(folder.name, subprocess.check_output(['git', '-C', str(folder), 'rev-parse', '--short', 'HEAD'], text=True).strip())

'''
Avoid reinstalling dependencies when this cell is rerun.
重跑此格時避免再次安裝套件。
'''
installed = ROOT / '.zimage_kaggle_v045_installed'

'''
Preserve Kaggle's CUDA-enabled PyTorch during dependency resolution.
安裝相依套件時保留 Kaggle 原有的 CUDA PyTorch。
'''
import torch

'''
Store the installed PyTorch version as a pip constraint.
將既有 PyTorch 版本設為 pip 限制。
'''
constraint = ROOT / 'zimage_torch_constraint.txt'

'''
Prevent dependency updates from replacing the CUDA build.
避免依賴更新時替換 CUDA 版本。
'''
constraint.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')

'''
Install ComfyUI, GGUF, and download dependencies once per release.
每個指定的 ComfyUI 版本只安裝一次相關套件。
'''
if not installed.exists():

    '''
    Install the official ComfyUI dependencies.
    安裝 ComfyUI 官方依賴套件。
    '''
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(COMFY / 'requirements.txt')], check=True)

    '''
    Install the GGUF extension dependencies.
    安裝 GGUF 擴充節點依賴套件。
    '''
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt')], check=True)

    '''
    Install helpers used to download models and fetch images.
    安裝模型下載與圖片傳輸套件。
    '''
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests'], check=True)

    '''
    Record successful installation.
    記錄本次已完成安裝。
    '''
    installed.touch()

'''
Confirm that installation is ready.
顯示安裝完成狀態。
'''
print('ComfyUI installed')

## 2. 下載模型


In [ ]:
'''
Import the official Hugging Face downloader.
匯入 Hugging Face 官方下載工具。
'''
from huggingface_hub import hf_hub_download

'''
Preserve the model files and formats from the original notebook.
保留原始 Notebook 的模型檔案與格式。
'''
MODEL_FILES = [
    ('unsloth/Z-Image-Turbo-GGUF', 'z-image-turbo-Q6_K.gguf', 'unet/z-image-turbo-Q6_K.gguf'),
    ('Comfy-Org/z_image_turbo', 'split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors', 'text_encoders/qwen_3_4b_fp8_mixed.safetensors'),
    ('Comfy-Org/z_image_turbo', 'split_files/vae/ae.safetensors', 'vae/ae.safetensors'),
]

'''
Download each model only if its destination is missing.
僅在模型檔案不存在時下載。
'''
for repo, name, relative_path in MODEL_FILES:

    '''
    Resolve the model's ComfyUI destination.
    取得 ComfyUI 模型檔案路徑。
    '''
    destination = COMFY / 'models' / relative_path

    '''
    Create the destination directory if needed.
    如有需要，建立模型目錄。
    '''
    destination.parent.mkdir(parents=True, exist_ok=True)

    '''
    Download missing model files without replacing valid ones.
    只下載缺少的模型，保留原有檔案。
    '''
    if not destination.is_file():

        '''
        Remove broken symlinks left by a previous Kaggle session.
        清除前次 Kaggle 工作階段留下的無效連結。
        '''
        if destination.is_symlink():

            '''
            Discard the invalid link before recreating it.
            移除無效連結，再重新建立。
            '''
            destination.unlink()

        '''
        Store a cached copy of the checkpoint on disk.
        將模型權重下載至本機快取。
        '''
        cached_file = hf_hub_download(repo_id=repo, filename=name)

        '''
        Link the downloaded file into the expected ComfyUI folder.
        將已下載模型連結到 ComfyUI 指定目錄。
        '''
        destination.symlink_to(cached_file)

    '''
    Show the available model name.
    顯示已準備完成的模型名稱。
    '''
    print('Ready:', destination.name)

## 3. 啟動模型服務


In [ ]:
'''
Import the tools for managing a persistent local server.
匯入管理常駐服務所需工具。
'''
import time
import requests
import torch

'''
Require the Kaggle T4 x2 accelerator profile.
確認 Kaggle 已啟用雙 GPU。
'''
assert torch.cuda.device_count() >= 2, '請在 Kaggle 設定中選擇 GPU T4 ×2。'

'''
Keep one server on a dedicated local port.
使用固定本機連接埠，沿用同一個 ComfyUI 程序。
'''
BASE = 'http://127.0.0.1:8189'
LOG_PATH = ROOT / 'zimage_server.log'

'''
Check whether the model server is already running.
檢查模型服務是否已經啟動。
'''
try:

    '''
    Probe the ComfyUI status endpoint.
    透過 ComfyUI 狀態 API 檢查服務。
    '''
    running = requests.get(BASE + '/system_stats', timeout=3).ok

except requests.RequestException:

    '''
    Treat an unreachable service as stopped.
    若無法連線，視為尚未啟動。
    '''

    '''
    Request a new server process.
    標記需要啟動新程序。
    '''
    running = False

'''
Do not reuse an old ComfyUI process after updating its source files.
更新原始碼後不可沿用尚未重新啟動的舊服務。
'''
if running and globals().get('ZIMAGE_CODE_UPDATED', False):

    '''
    Require an explicit restart for processes not owned by this setup cell.
    服務可能不由此 Cell 管理，因此要求先關閉舊程序。
    '''
    raise RuntimeError('ComfyUI 程式碼已更新；請先停止 8189 連接埠的舊服務，再執行本 Cell。')

'''
Start ComfyUI only once, with retained model caches.
只啟動一次 ComfyUI，並保留模型快取。
'''
if not running:

    '''
    Write server diagnostics to a persistent log file.
    將服務執行紀錄寫入 log。
    '''
    log_file = LOG_PATH.open('w')

    '''
    Keep models on their assigned GPUs and cache loader results.
    讓模型保留在指定 GPU，並快取模型載入結果。
    '''
    server = subprocess.Popen(
        [sys.executable, str(COMFY / 'main.py'), '--listen', '127.0.0.1',
         '--port', '8189', '--highvram', '--cache-classic', '--preview-method', 'none'],
        cwd=str(COMFY), stdout=log_file, stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    '''
    Close the notebook's handle; the subprocess owns its copy.
    關閉 Notebook 的檔案控制代碼，讓子程序繼續寫入。
    '''
    log_file.close()

'''
Wait for ComfyUI to complete startup and register custom nodes.
等待 ComfyUI 啟動完成並註冊自訂節點。
'''
for attempt in range(120):

    '''
    Check whether the server responds.
    檢查服務是否能正常回應。
    '''
    try:

        '''
        Stop waiting when the server is ready.
        服務準備完成時停止等待。
        '''
        if requests.get(BASE + '/system_stats', timeout=3).ok:

            '''
            Continue to node validation.
            接著檢查必要節點。
            '''
            break

    except requests.RequestException:

        '''
        Retry when ComfyUI is still starting.
        ComfyUI 尚未啟動完成時繼續嘗試。
        '''

        '''
        Ignore temporary connection errors.
        忽略啟動過程中的暫時性連線錯誤。
        '''
        pass

    '''
    Avoid excessive status requests while starting.
    避免啟動期間過度頻繁查詢。
    '''
    time.sleep(2)

else:

    '''
    Fail clearly instead of silently continuing with a dead server.
    若啟動失敗，直接顯示 log 位置。
    '''

    '''
    Raise an actionable error message.
    顯示可供檢查的 log 檔案。
    '''
    raise RuntimeError(f'ComfyUI 啟動失敗，請檢查 {LOG_PATH}')

'''
Query the registered ComfyUI nodes for both GPU loaders.
確認雙 GPU 模型載入節點已成功註冊。
'''
node_info = requests.get(BASE + '/object_info', timeout=60).json()

'''
Specify the required multi-GPU loader nodes.
列出雙 GPU 模型載入需要的節點。
'''
required_nodes = {'UnetLoaderGGUFMultiGPU', 'CLIPLoaderMultiGPU', 'VAELoaderMultiGPU'}

'''
Detect missing custom nodes before submitting a workflow.
在生成前先檢查是否缺少自訂節點。
'''
missing_nodes = required_nodes - node_info.keys()

'''
Avoid silently falling back to one GPU.
缺少節點時直接報錯，不默默退回單 GPU。
'''
assert not missing_nodes, f'找不到雙 GPU 節點：{sorted(missing_nodes)}；請檢查 {LOG_PATH}'

'''
Show the active GPUs and the retained server connection.
顯示 GPU 配置及已啟動的常駐服務。
'''
print('GPU 0:', torch.cuda.get_device_name(0), '｜影像生成')
print('GPU 1:', torch.cuda.get_device_name(1), '｜文字編碼與 VAE')
print('ComfyUI ready:', BASE)

## 4. 生成設定


In [ ]:
'''
Describe the desired image.
設定圖片內容。
'''
PROMPT = '''

'''.strip()

'''
Set image dimensions and seed (-1 makes a fresh random seed).
設定圖片尺寸與種子（-1 代表每次重新抽取）。
'''
WIDTH, HEIGHT, SEED = 640, 960, 42

'''
Preserve the original Turbo sampling parameters.
保留原本的 Turbo 採樣參數。
'''
STEPS, CFG = 8, 1

'''
Choose the PNG filename; duplicate names get numbered automatically.
設定 PNG 檔名；同名時自動編號，避免覆蓋。
'''
OUTPUT_NAME = 'zimage.png'

## 5. 生成、預覽與下載圖片


In [ ]:
'''
Import tools for tracking and displaying the generated image.
匯入追蹤進度與顯示圖片的工具。
'''
import uuid
from IPython.display import FileLink, display
from PIL import Image

'''
Check image dimensions before submitting a job.
檢查圖片尺寸設定是否有效。
'''
assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 64 == 0 and HEIGHT % 64 == 0, '寬高須為正整數且為 64 的倍數。'

'''
Generate a fresh 32-bit seed only when -1 is requested.
只有設定 -1 時才建立新的 32-bit 隨機種子。
'''
import secrets

'''
Resolve one seed per run without overwriting the notebook setting.
每次生成決定一個種子，但不覆寫 Notebook 的原始設定。
'''
RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

'''
Require a supported seed before building the inference workflow.
在建立推論工作流程前，先檢查種子範圍。
'''
assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'

'''
Show the actual seed so a random run can be reproduced later.
顯示實際使用的種子，方便之後重現隨機生成結果。
'''
print(f'本次生成 seed：{RUN_SEED}')

'''
Define the same GGUF workflow with its loaders fixed to separate GPUs.
保留原本 GGUF 工作流程，並將不同模型固定分配至兩張 GPU。
'''
workflow = {
    '1': {'class_type': 'UnetLoaderGGUFMultiGPU', 'inputs': {'unet_name': 'z-image-turbo-Q6_K.gguf', 'device': 'cuda:0'}},
    '2': {'class_type': 'CLIPLoaderMultiGPU', 'inputs': {'clip_name': 'qwen_3_4b_fp8_mixed.safetensors', 'type': 'lumina2', 'device': 'cuda:1'}},
    '3': {'class_type': 'VAELoaderMultiGPU', 'inputs': {'vae_name': 'ae.safetensors', 'device': 'cuda:1'}},
    '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': PROMPT, 'clip': ['2', 0]}},
    '5': {'class_type': 'ConditioningZeroOut', 'inputs': {'conditioning': ['4', 0]}},
    '6': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': WIDTH, 'height': HEIGHT, 'batch_size': 1}},
    '7': {'class_type': 'ModelSamplingAuraFlow', 'inputs': {'shift': 3, 'model': ['1', 0]}},
    '8': {'class_type': 'KSampler', 'inputs': {'seed': RUN_SEED, 'steps': STEPS, 'cfg': CFG, 'sampler_name': 'res_multistep', 'scheduler': 'simple', 'denoise': 1, 'model': ['7', 0], 'positive': ['4', 0], 'negative': ['5', 0], 'latent_image': ['6', 0]}},
    '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['3', 0]}},
    '10': {'class_type': 'SaveImage', 'inputs': {'filename_prefix': 'zimage', 'images': ['9', 0]}},
}

'''
Submit the image job to the existing ComfyUI server.
將生成工作送往已啟動的 ComfyUI，不重新載入模型服務。
'''
response = requests.post(BASE + '/prompt', json={'prompt': workflow, 'client_id': str(uuid.uuid4())}, timeout=120)

'''
Surface validation errors instead of silently waiting.
若工作流程驗證失敗，立即顯示錯誤。
'''
if not response.ok:

    '''
    Show the response body to help diagnose node or setting errors.
    顯示伺服器回覆，協助定位節點或設定問題。
    '''
    raise RuntimeError(response.text)

'''
Save the submitted job identifier.
記錄本次生成工作的編號。
'''
job_id = response.json()['prompt_id']

'''
Track the elapsed generation time.
記錄本次生成所花時間。
'''
start = time.monotonic()

'''
Show status while waiting for the PNG output.
等待 PNG 輸出時定期顯示進度訊息。
'''
for attempt in range(1800):

    '''
    Read the current workflow history.
    讀取目前工作流程的執行紀錄。
    '''
    record = requests.get(BASE + '/history/' + job_id, timeout=30).json().get(job_id)

    '''
    Inspect completed or failed jobs.
    檢查已完成或失敗的工作。
    '''
    if record:

        '''
        Report execution errors from ComfyUI.
        顯示 ComfyUI 實際執行錯誤。
        '''
        if record.get('status', {}).get('status_str') == 'error':

            '''
            Stop when the server reports an error.
            服務回報錯誤時立即停止。
            '''
            raise RuntimeError(str(record['status']))

        '''
        Read image metadata when generation finishes.
        生成結束後取得圖片資訊。
        '''
        images = record.get('outputs', {}).get('10', {}).get('images', [])

        '''
        Stop polling as soon as the image is available.
        圖片已可取得時結束等待。
        '''
        if images:

            '''
            Continue to downloading the image.
            接著下載生成圖片。
            '''
            break

    '''
    Print periodic status in the cell output without assuming a step percentage.
    定期顯示已等待時間，不捏造採樣百分比。
    '''
    if attempt % 5 == 0:

        '''
        Display elapsed seconds while the server is generating.
        在 cell 下方顯示目前耗費秒數。
        '''
        print(f'生成中… {time.monotonic() - start:.0f} 秒', flush=True)

    '''
    Pause before the next history request.
    等待片刻，再次查詢工作狀態。
    '''
    time.sleep(2)

else:

    '''
    Stop if the server never returns an image.
    若長時間沒有產生圖片，則回報逾時。
    '''

    '''
    Report a clear timeout rather than looping indefinitely.
    顯示逾時訊息，避免無限等待。
    '''
    raise TimeoutError(f'生成逾時，請檢查 {LOG_PATH}')

'''
Download the completed PNG from ComfyUI.
從 ComfyUI 取得生成完成的 PNG。
'''
image_response = requests.get(BASE + '/view', params=images[0], timeout=120)

'''
Reject failed downloads to prevent corrupted files.
防止下載失敗卻仍儲存不完整圖片。
'''
image_response.raise_for_status()

'''
Sanitize the requested filename and enforce PNG output.
清理輸出檔名並統一儲存為 PNG。
'''
filename = Path(OUTPUT_NAME).name

'''
Ensure output filenames always have the PNG extension.
確保輸出檔名使用 PNG 副檔名。
'''
if not filename.lower().endswith('.png'):

    '''
    Append the standard PNG extension.
    自動補上 PNG 副檔名。
    '''
    filename += '.png'

'''
Save results in Kaggle's working directory.
將結果儲存在 Kaggle 的工作目錄。
'''
OUTPUT_PATH = ROOT / filename

'''
Start suffix numbering for duplicate filenames.
從 002 開始編號同名圖片。
'''
index = 2

'''
Preserve earlier generations by choosing an unused filename.
選擇未使用的檔名，避免覆蓋先前作品。
'''
while OUTPUT_PATH.exists():

    '''
    Add a numeric suffix for each additional image.
    在重複檔名後加入編號。
    '''
    OUTPUT_PATH = ROOT / f'{Path(filename).stem}_{index:03d}.png'

    '''
    Move to the next number if this name is already used.
    若名稱仍重複，繼續嘗試下一個編號。
    '''
    index += 1

'''
Save the new PNG to Kaggle storage.
將生成的 PNG 寫入 Kaggle 儲存空間。
'''
OUTPUT_PATH.write_bytes(image_response.content)

'''
Show completion time and the saved filename.
顯示生成耗時與儲存檔名。
'''
print(f'完成：{OUTPUT_PATH.name}｜{time.monotonic() - start:.1f} 秒')

'''
Preview the finished image in the notebook.
直接在 Notebook 中預覽生成圖片。
'''
display(Image.open(OUTPUT_PATH))

'''
Provide a local link to the saved PNG.
提供 PNG 檔案下載連結。
'''
display(FileLink(str(OUTPUT_PATH)))

## 6. ngrok 與遠端生成 API 啟動

In [ ]:
'''
Dependencies are available in the Python standard library.
只使用 Python 標準函式庫處理遠端服務與檔案。
'''
import zlib
import math
import re

import queue
import copy

'''
Use secure randomness when the API requests seed=-1.
API 指定 seed=-1 時，以安全的隨機來源產生實際種子。
'''
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

'''
Expose only the inference bridge, never the Jupyter or ComfyUI management interfaces.
ngrok 僅轉送推論橋接 API，不公開 Jupyter 或 ComfyUI 管理介面。
'''
REMOTE_PORT = 8765
REMOTE_SERVICE_ID = '06_Z-Image-Turbo_圖片生成_Kaggle'
REMOTE_KIND = 'image_text'
REMOTE_FIELDS = ['task', 'prompt', 'width', 'height', 'steps', 'seed', 'cfg']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda}Yko\x13W\x1a\xfe\xce\xaf8\xd2\xaa\xb2\xdd\x82\x9b\x0bP@\xcdJl\t!*\x97,\t\xbb\xd2"d\x8d\xed\xe3x\x9a\xf1\x8c;\x97\x844\x8ad*B\x12\x1c\x93l!\xe1\x96\x12\xc2%5PH\xa1 B\x08\xe1\xbf\xecz\xc6\xf6\xa7\xfe\x85}\xdesf&coh?\xd0\xcc\xcc9\xe7\xbd?\xef\xf3\x1e\xff\x85\xf5\x17\x8a\x86i3\xdb04\x8b\xe5\x0c\x93\xd9\xa6\x92\x19Q\xf5a\xa6\xe8Y\x96U\xad\xa2\xa6\x8c\xd3\xa3\x9d\xe7l\x98\xeb\xdcTl\x9eejA\x19\xe6\xc9=\x7fa\xee\xdc\xba;\xf5\xb8\xf1\xf1C\xe3\xdd\xa3f\xe9\x95\xbb\xb9\xd6\x98\x99n\xae\xae\xd7\x1fm\xba\xcbK\xf5\xd9\xe9\xfa\x9d\xcb\xee\xdb\xc7\xee\xd4\xdb\xff\x96~\xdc\xa3Ji\x8e\xa3f\xf7\xe4L\xa3\xc0\xfa\x07\xc6\xed\xbc\xa1\'}A\xcc_p\\\xd5\xf8IU\x1f\xd9\x1bh W\x0f\xf4\x9f\x0cV\xf4\x93\x02{\xa0\xc07y\x9e\x19\x91\xfa`q\x81\xeb\x96j\xe8\x16Ks\x18\xc3\x99\xe5\xa4\x0b\xaam\x0b{\xd8wF\x9aT\xf6\x9e>\xf0V\x1eK\xed\xdc\xdf6\xdd\xf5\x8dF\xf5\xb9\xfb\xe2\x8ewk\xdd]X\xf3\x96g\xbd\xc5\x19RV\xb1,\x0eI\xff\xec?6t\x82\xfd\x95u\x08\x8f\x9c\xe8\xed\xef;1\x14>\xca\x8f\x9f\xb1\x83\xfbYOO\xeb\x92\xf0\xe5^\x16s\xd7\x7fm>\xbb\xd5\xbc?S\xffq\xd3{\xfe\xd0[|\xed-n\xd46n\xe0\x91V\x91\x8fJ\x15\xbc\x82\xd8\x18\x19\xd5\xe7;\x1aJ\xe7Ln\xe5Yw\xd7\xbe\xb4j3\x8b\xc3\xf7\x86\xae\x8d\xb3\xb1<\xd7\xd9\xbeN\xa6Z\xcc\xe4\xdf;\xdcBTD<\xe6\x9f\xc2\x02i\x10}\xf7n\xff\xe8\xcdV\xdc\xf7\x9b\xf5geo\xe97H\n\xcej\xde\xa9zOV\xea\xd5\x17\xee\xf3\x85Hl,\x9e1\xb9m\x91\x16g\xb9eh\xa3\x1c\x02\xb9\x94\\\xe4&3\x1d\x9d\x8d\xa9\x08\x9ac3c\x94\x9bc\xa6j\x07\xf9\xa1\x1b6O\x1b\xc6\x08V\x0b\x9f\x0bo\xaf\xcf{\xbf\xae\xd6o\xacx3\x0b\xde\xcbM\xa8U\xdb(\xb9\xa5\xb2\x14\xfc\xc7\xd6\\\xed\xc3\x95\xdaF\xa5\xb1v\xc5]\x7f\xc6N\x07\'\x90K\xae\xad\xb8\xbf\x94\xa5)\xa4\xe0\xd9s\xa7S\x83\xbd\xbd\xc7XO\xa0d\xd2\x84\xbf\xd3\\3\xc6\xe2]\x9f\x7f\xde\xdd\x95`j\x8e\xc9%=d<\xd7,\xceT\xdd\x8e\xd3\xbb\x844\xe9{G5\xc9\xaf\x96S${a\x950\xcd\xcf\x97\xb4\xa3j\xd9\xc0\x1cU\xcfq\x93\xeb\x19\xce\xc6\x0cs$\x071\xc2\xc3\xcbU\xdf\x9d\xd7\xaa\x8dgKH\xee\xda\x87e\xef\xcd\xa5z\xb5\xec\xceV`\x90;5#SL\x9aX_\xbf\xec.W"\t\xd5\xc1\xbe\xeea\xa11_3\xa1:\x92D<\xba\x1f\xa7\x90&\xb5\xf7\x08\xd6\x16V6\xa6\x7fg\xfb\xbb\x0e\xef?|\xf0\xab\xae\xc3\x07p\xb87\xb3\x04\x8fP\xde\xec\xeb\x0c\xb2e0o\x8c\t\x85\x95\x8c\xed(\x9a4\xc82`$\xf9\x07uC1\xcb(:\x8cD\xb2\x14M#\xebd\xb0BC\x82\x99d\x91_\xae\xebk\xcd;wk\x1f>\xd6oT\xe1\xfd0>\xde\xd2\xbb\xda\xf6\xc7\xda\xbb\xb2\xbb=\xd7\x9c\xae\xd4\xe7\xb7\xfd\xcc\x11!\xad\xbfY\xf0\xee-\x93uE\x93<\x9d\x8by\xcb\xbf\x86\x01\x17\xaa\xfc\xb1ug"0w2&\xc2p\x8c\xe7T$\x15\xe9l)\x05\xce\xfa\xfa\xce\x1d\x0f\x9d,\xd2\x8b\xa9\xb6\xc54C\xc9r\x13\xb8\xa4^\x84\xbe\xb6\x81\xe3\x8a\x8a\xa8\x8b\xbe\x81s\x16\xa9^\xfb\xf8s}\xf162\x05R\xe5)\xd1xPvm\xac\xb9\xbfM#\xc1\xdc\x859\xaf\xba\xea\xde+\xbbw)\t\xdd\x99+\xcd\xa9\n\xdc\xebN=q\xb7\xde\xd0\x81dD\xa8C\x0f\x9b\xd8\xc3\xf0_\xac3v\x84M\xc42\x1a\x82\x97\xb2\xc7\x8b\x1c\x8f\xb1s:\xb7O\n\xddH\xe4)G\xb3U\xec\x8f!\x86\xaa^tlKlq\xb0(\xa5\xc3:\xda\xf1\xc3>\x81P\xfbl\xc7L\x1b\xfb\xfe~0\xf5mrx\xd8\xc9\xd1\x96,\x1fU3bQ\xc6\xc9*G:b\x93\x93{\xa5\xe8\xae]D\x7fs\xb2\x7f@\x8a\xfe\x84\xd8\x8c\xa6\x16C\xb1\xdf\x8fq=\xd5\x9d\xda\x9fN\xe5\x8a\x87R\x05rc\xd2Rr\xdc\x06L\x1a\xa6E[\x83\x835\xa7\xa0\xeaJ\xd7.\x1au\xeeh\xd4\xbd\x8bF\xff8\xda\xfb\xa7\n\x8d*<\xd4G\xe1\xed\xe2?-k\xff\'\xac\x1f\xe2\x17\xed^=cdy\x9b \x1b\x1f\xf0\xc7\xc0\xd93\xa7\x06\x86\xf0\x89<\x81\xe7\xf312\xaa\xe3Bx\xf0\x81\xdd\x0e6\xf4,p\xcc\xd0Q\xfb\xff\xe2\xa6q\xc6\xb1\xdb\xfd\x1aY!N\xdd\xdfz\xea\xc1]N\xed-\x14\xed\xf1\xc1c\xdd\'\x91\xb2\xba-ZV\xdb\xa9cj\xd6\xce\xe3/\xd1G\xf0)\xcf\xd5\xe1<Y!\x9b\x08\xde\xa4\x15;\x93OY\xea\x0ftbg(\xee\xab]\xc4\x9d\x82O\xb4A\xa5P\xd4\xa0\xe3Q\xc7T\x8e#\x93\xdb\x04Zy5G\xe7w\xe3u\x81\xd6\x0b[:[m9\xb4\xcb\xe1\xdf\x8a\x83\xb9\xd9~\x1e\n\x1c\x7f\x04\xf5\x8d\x8f\xe8DE\xfa68\xd4;0Ha\xc8\x91\xbf\xbe9\xdeG\xdf\xe4\x19a>\xa0\xb7\xa5\n\x946\xb4\x89N\xb62y\x9euH\x0c\xbeZ*\xad\x96Y\xa2\x1b\xaa%<\xd0\xa2\xf7WBo\xbc*\x1a\x16\x823\xca#\x91\xc1[\x9d\x0f+\xe1\xdb\x03\xc1[MD#%\nR|9\xd8j\xfd\xe1\xdd\x93\xfc\x18\xdf%\xe7\xa4A\x968\xe6P \x00\x19/^t\xb7\x9e\xdb\xd9\xb1\xcb\xc1\x83\xca(\xdf-1r`@\xe4\xa5T\xd1\x04V^\x14\x18\xa2\x86\xeb\xe8\x0f)\xf4p(cR\xb4\x02\xc1wd\xf7\x12\x94\x08\x9c\x87\x80\x93^\xf0\x8bp3\xf5\xb6o\x8cBn\xfc\\?\xd0\xd4\x1c\x95\x1d\x00\x10)\x01[\xa2g\xb3t\xc9\xdd.\xb9o_\xb9\x8b+ny\x91H\x83\xbfG j\x05m\x00\\\xa2\xb15\x0b\xea\'A\xd5[\xae\xb8\xe5\xdb\x84\xa0\x08i\x11\x04\x8c\x03A}bb%\x11\x1d;\xfe\xb7\xa3\x83\xbd\xec\x0b\x16\xfb\x12\r\x08u\x01\xbd\xbf\xb3\x0c\xbdg"\xe6?\x1f\t\x1b\x80\xac]\x11#\xca-\xcb6\xe3\xc4\x18\x93\xf4\xcf\xfex"1\xb9\x97\xd9`z` =\x9d]\x1d\ti\xb7\x99S\xd0\xa8G\x15M\xcd*T\xa6\x8c\x9b&0\x06M\x1c\xb9\xa5\x805\xe5\x98E>\xb5\x89<)j@O\x1a\xe5\xc7\xd1\x8e\xd1|r\xb3\xf1|\xd6}\xf4\xd2[\xbc\t[\xd1\xe5\xdd\xca\xef\xb2?6\xe7\xd6\x1bO\x1f\t\xa6\x94#\xa6\xc3\x02K\x93\xc6\xc8\x11\x11\xe1H\'\x0e\xbd\x906\xb2\xe3\x14\x80<\xd7\x8a\xe0\xa7\xca\xb0nXD\x94\xb2\xa0Uf\xc0\x93|]\x93\xfe)R^mk\x93\xbcz\xbb\xea\xde\xbd\x07vD\xc4\xa2r\xc3\xbd\xfa\x84\x18\xd4\x87J}\xbd\xd4|\x7fO\xb2\x01jg\x8b\xf3\xcd\xd59R\x8e\x8e0\x15\x14\x0b;\xeb\xe8\xe4\xa6^:;\x1e*K()]\x86\xcc\x93\x1dXrdtXJ\x16\x15\xb5f\xab9U&F\xa3z\xab9w9\xda\xd2\xa5\xb7\x88 \xbc\xad6n\xaf\x90HlC\xa4D\xc0}!\x14\xd9x\xe2\xbc\x1fZ\n\xe3\x05\x129D\xb3\x85\xccEM)Z\x90\xe8O\x14\x14/\xd2uw\x91\xdel\xa9q\xf5%\x98ls\xe9:\xc9\xb3l\x054\xaaG\xee(\x18\x88\x05P9\x13O\x84\\\x08\x0bl\xc7\x02GF\xc0\x83`\xcb\xe1\x06\xa2\x07N\xf71\xa4\x0e*\x8d\xa4\xd5\x11\xed\xed)\xf1\xb2\xb1\xb5\xe1NoB\x0c\xcd\x02\xcb+~\xd4\xe5LS\xbd\xea]Z\'\xd9t\x8a\x02g\xc1,\xa4\x16\x11\xaca\x1e\xef<\xd4\xd1\x91\x08r\xe0,e\x1b\t\xca8&\xd8\xa3\xbd\xc3k\xf2\xa8?\xc3\x1c\x0f\xc2\xdcxQr\xe7\x97\xeaw_\x806Fs\x90\xa8\xef\xca\xdb\xc6\xea\\\xfdu\t\x9e\x08\x83\n\xfc1\xb3\xd1\xb2\x1a\xe6\x91\xaa\xf2\x0f\xff2\x86\'\x19\x90\x9d*\xe9\xeeH\xf8!\x11{\xe4g\xb8K\xea\xd1\xaf[E\x9e\xb1Y\xc6 ,\xa3<\x80\x8d9\x05\xbe\x13\x19\x11\xa6\xa5?7\x01\x15^\xccQTf\x96d\x9d\xc8\t\x0f\xea\x07\x9a\xa2>\xa4\xb2\xd2%\x81[\xc4x\xc1/\xf2\x8c\x13\xadO1\xd8\xf9\xe8\x92\x8c\xac\x97\xee\x0f\xb1J\xb2S\xe9\x97\x9dJ\x0c\x96\x87\x12\x85}1\x19\x7f \xcc\xc4d"\xfa&\x05(\x89%h8\x88\t\xe9\xb1\x1d\x05\xfd\xfa\xb5\x8d\xa2\x1c\xadDa\x08|$\xca\x0c\xcd-\xccvR\xe7d\xdb&\x89}\xa8R\xf7\xfeK\xa9\x1arH"\x87{i\xd9{\xfe \xaa\xe9\'\xca\x930NZp>\xd0\xfeB\xc2\x8fP$\xad$\xa6\x17\xb8\xad\x00\xe4\x14\xa9i\xa4\x84@\xaaU+\xcf\xad\xa8\x1fwh\xfa\xcf+`\xf0\xc88w\xfb\xa6\x1c~\x1b\xbfO#\xb1[\xfc(\x1a\x8b\xc8\xb1\x1do\xcabiq\'ZY\xe4\xc9oG{\xd9\xf9\x0b-*\x0bo\x16\rM\x13\xe3\xb7\x85\x91\x04:\xe2\xff;\xdd\t\xd3\xab2\x8aLS\xd2\x1a\x8f*\xed\x0f\xe7H\xb5\xf9u\xa91\xb9T\xd8 \x0b\xb6-\xf8R\x83\xf6h\x82\xd4\xa1\xf2\x1dN\xf8\x9b5\xc6t\x1a\'\xc2\xd9N^`\xb4E\xf2\xda\xe3\xc6\xbfo\xd66\xcahm>\xda\tE\xda\x03\x986\xb92\x12\x94\xcf\x00\xcd=4\x1a\xabFV\xcd\x04\xd8\xa3\xca\x14\xcapM\xf3\xd1&\x9c\x9a\xd1\xfe\x89k\xd3\x8d\x04\xd1\x1e\xda\x9a\x01N\x84\xfa\xc0\xfc\x08\x00\xc1\t\xd2d\x89\x7f\xb2\x01{\x95\xf9f\xe9\xbewm\xd5\xab>\xac\xdf\xde\xc60\xe3\xad\xdf\x88\x94_\x80P\x9f\xb1\x03\xe2\n"Z\x88\xc7\xfc\xab\x96\x00\x821K\x83\xda\x06h\x19\xc9{\x04\'\xc8-\xea\x95\xd1\xe8T\xa5a\xf0\x14\xa6C\xa9\xa7\xc4\xb1F\xe9f\xe3\xd5\xfb\xfa/?\xc9\xcb\x8cpS0\x1cJ\xaf\xd66\x9e\xff\xa7\xb4\xc6&\xda\x01\x9c\xedc\x02\xdb\x8f$;r\x93\x0c\xa7 \xa7r\x9ac\xe5{\x86L\x87\x87\x805\xa08\xd4Y\xe5\xc4.\xee\x1d\xd0\xd3\x02t\r\xe01\xf0\xa5t\x1ee\xd3\xcc{\xea\xa1W*\xe8+\xc0\xb1\xc6\x93\x07~++\x97\xbc\xa9r\xa0\xacP\xc9\xd28/\xc6\xbbdC\xa1$\x86K#\x8e\xd1\xb9\x84\x05\x0cs\xba\x80\x85\xf06\x0cl\xa2\xb9\xf8VF\xca{\xf5\x93\xb7\x8c4z\x00\x9be\x1a\x91x\xe8!\x81\xa2\xb4\x8de$\x95.)vz\x87\x00I\x85e4\xae\x98\x01~\x030 \x9d\xba\x17di\x86Q\xa4\xe4Q\xf5,\r\xd1\xaa\xcd\xb5\xf1V\xe2 \x8f\x96-\x0b"\x9b\x97>\xbaS\x95\xfa\xe5\xd5\xe6\xed\x85\xd6\xe2\x9184$\x85H\x1c\n"$\xcf\xc0\xee\xc6\xb3\xb2\xc4}6q\xf2L_j\xe0\xe8\xd0\x89`\x86\xf7+J\xe6y\xd8;\xa8\x93\xb6b:\x12f\xfb\xde\x0e\x90\x8br\xf6\xabK4\x12\xa2\x97\xd8%\xef\xa0\xe0\xc8\xd4n\x14\xb2\xa5\xd7\x8d\xaa\x9c\xe6\x1a\xba\x0b(X=\xb2\xf8\xcf\x13\xf5n\xa7\x85g\xf9w\xd4\xd9\xfcf\x16`\x80E\x80\x00^=J\xdd\x198g:E\xd2\x9c(\xb7\xb8Th\xdez\x05\xd4\x960 [\x9c[y_{_q/\xbfr\x9f\xdf\xa2\xbb\x04\xe8\xbd\xf8z\x07\x1bZ\xd5N\n\xbf\xa6\x90\x9d)\t\x06>3Q\x10,\x8cq>?\xf4\xef\xecX@\xf4\xc5\xd5!\xd7\xb1+\xd3\xceQ\xbc\x8d\xa9\xfa\xc2\x15\x9f\x9e<\xbd\xe1.Tj\x1bk\xf57/\xe9"M\xa8D\xd7A\xbe\x07\xc3\xe3zP&v>~\xe6\xdc\xd0\xc0\xb9\xa1\xd4\xe9\xa3\xa7z\x13I\xfa@\xaa\xf4\xea\x96\x83\xd2\xf1q)\xd8\x82T\xd6\xc6\x94q\x8b\xe5\x03fHj\xa0\xb4\xe4\x9d\xaa`K\x0f6k\x1f\x7fn\xd1D\xdc\x17\x89\x95\xee\xec\xba|\x17\xe1\xc8\xc1\xd9I\xf0\x1fn\x82\x7fp@\r\x01a<\x96,b\x90\x0eY\xd3\xd1b\x11\x9fd\x95\xd9p\x85bf\xdb\xc5\xfb\xaci\xfa)F\x92\xc6\xc3\xe5\xda\xc6\xd5\xff\x17KkB\x17|\x81>/\xa4\x84t\x17!\xc2\xb4)\xd0\xf9[ex\x18\xc3\xa5%\xe8\x19\x15TVE\xd3\x93\xfcL\x0eD\xe2fK:\x98\x10On`!\xdb\x01\xe0\xf9\xec\xcc\xf71\xd5\x06\xdc~\xf6\xcc\x99!\xf6e\xa8\x85\xc4\x10"\xac\x96\x93\xc3$\xc7t\xa7\x90F\xb3\xf0\x19i\xd6\xc1\xb4\x9e\xa1+\xad0\x0cA\xc5ttt\xb1\xe6R\xd9\xfd\xa5,\xa9\xb6\xbb0\x07+#Y\x07\x04\xb8\x08\x81]$b\x00\xa6\x11:1\xe0\x06f\'3B\x0b,\x96\x1eg\x99\xbc\x81\x11Y\\\xeb3Gw\xacH\xe6\x89\x94\xbf\xb4\xe1]\x9f\xf6\x96\x9f\x86\xf7\x7f\xd2\xa7!~`\x02i\\/\xbbS3\x00y\x18\xef^\xbf$\xae\xcbD\xcf\x88\xd8\x9f\x14C&2>\x8ck6\x0b8\x83\xcd09\x13\xb8\x80\x0c\xe7J&\xcf\x94\xac\xbcSQ\xb4hC\x16\x1d\x063f\xe3\xd1\xb4\xd4\x82x\xcb\xd5\xfb\x186wf\x0eZ\xb7\xbb\xdfc\x13"\xf1\x03\xeb\x12I\x94Ya25!\xdcu\xa4\xa3;;\xe9\xe7\x84\x14u\xca\x18\xe5\xc1\x84,\xfa\x88\x8c\x8f\x84|\xb4@\x91I\xc4S44}\xccr\xe4\xba0\x171<.T\xea\xd5\x97\x00\x07\xa9/\xcd\x8c\xef\xb6\xea\x9bs\xee\xad\x9b\x8d\'k\xc0\x10\xff\xb6\xbbEq\x199\xa4gg\xcb \xa6\xf31\x91\xd1\xd0\xc6O6\xeah\xc1\x0f-\xc1\x88\xee\xc3%\x18\xf13x$X\xe9\x03\xc1\x93M\x7fF\x8a\xc6\x84\xae\xe8y*=ns+\xde\x06Uh\xfdt\x19\xb23:\xf9@\x1e\x0cd\x02\x95\xe45\xedh{\xca\xc8\xb6/tB\xdb\xa7\x9633-\xd5\xd8\xa9\xc7\xa0\xefK\xa4\xa7\x9b\xe0\xa8bt\xd4\xe4\x1f\x1f\x96\xff\x84\x08t\xfaD \xe1\'9A\xbfP\xc8\xe7\xba\x01\'\xf6\xf9V\xf0\x0b\x84\x00\xab\xbb\xafA\xe8\xa8t\xc3_\x15@<\x9a\xf7\x17\x1ak\x1f\xda\xd9\x9d\xff3S\\\\\xc5$\r`Q<\xa2h\xc2\x97n\x8cb:F:kF\x06\x19\x0bZ;\x12d\x8et\x0f\xc2"\xf0z~\xa1\xb6}W\x04\t\x9e\xf0Vgd+i\x96\x1e\x02S\xa2\xd2\x82\x9f\xb9\x04\xfdo\x91\x97\xf8\x1f\x91\x8de\x06').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['text']
REMOTE_MAX_IMAGES = 0

'''
Keep job directories isolated between different notebook services.
各 Notebook 使用獨立工作資料夾，避免不同服務互相干擾。
'''
REMOTE_ROOT = Path('/kaggle/working/remote_api_jobs') / REMOTE_SERVICE_ID
REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('REMOTE_ENGINE_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=32))
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(2))

'''
Results remain available for up to 24 hours, subject to an 8 GiB storage limit.
結果最多保留 24 小時；若超過 8 GiB，會提前清除最舊結果。
'''
REMOTE_RESULT_TTL = 24 * 3600
REMOTE_RESULT_BYTES = 8 * 1024**3
REMOTE_MIN_FREE_BYTES = 2 * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = 2 * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'Kaggle CUDA GPU 不可用；請啟用 GPU 並重新執行模型初始化 Cell。', []

        '''
        Use the model's actual GPU requirements, not the number of queued jobs.
        依各模型原本的 GPU 配置檢查，不因佇列工作數增加 GPU 使用量。
        '''
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [0]
        else:
            selected_gpus = [0, 1]
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]

        '''
        Validate resident PyTorch pipelines and their designated CUDA devices.
        檢查 PyTorch Pipeline 已初始化，且核心模型使用指定 CUDA 裝置。
        '''
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != 'cuda:0':
                return False, 'Qwen Transformer 沒有設定為 cuda:0。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, 0), ('HeartCodec', pipeline.codec, 1)):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            '''
            Query the actual ComfyUI process; a running HTTP server alone is insufficient.
            查詢真正的 ComfyUI 裝置資訊；HTTP 服務存活不等於正在使用 CUDA。
            '''
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐顯存）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task

    '''
    Confirm that the requested video task matches the weights already loaded.
    確認影片模式與目前實際載入的模型權重及量化設定一致。
    '''
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 程序已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))

    '''
    Check numeric types, finite values, and limits used by the generator.
    檢查數值型別、有限值與生成程式的實際有效範圍。
    '''
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]


'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return

        '''
        Reuse the same job ID for retry attempts after an uncertain response.
        網路回應不明時，重試會沿用同一個工作 ID，避免重複生成。
        '''
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)

            '''
            Reject invalid parameters and incomplete model configurations before queuing.
            在排隊前拒絕無效參數與尚未完成初始化的模型設定。
            '''
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):

                '''
                Ignore missing seeds and validate the supported 32-bit range.
                未提供的種子沿用 Notebook 設定，並檢查 32-bit 範圍。
                '''
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')

                    '''
                    Translate -1 into an explicit random value before inference.
                    推論前將 -1 轉成明確的隨機數值。
                    '''
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)

            '''
            Save supplied seeds for reproducibility in the job status API.
            在工作狀態 API 中記錄請求所使用的實際種子。
            '''
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        '''
                        Point R2V to this job's uploaded reference images only.
                        讓 R2V 僅讀取這筆工作的參考圖片資料夾。
                        '''
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:

                    '''
                    LTX-2.5 uses a shared ACTIVE_JOB configuration.
                    LTX-2.5 使用共用 ACTIVE_JOB 參數。
                    '''
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'

            '''
            Clear old output metadata so a failed generation cannot reuse another job's file.
            清除前次輸出狀態，避免本次失敗時誤取其他工作的結果。
            '''
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)

            '''
            Execute a fixed local generation cell, never user-supplied code.
            僅執行 Notebook 內固定的生成程式，不會執行請求傳來的原始碼。
            '''
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):

                '''
                Include YuE2 replay metadata and score with the single MP3.
                將 YuE2 的重播參數與 ABC 樂譜一起提供下載。
                '''
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

'''
Keep one worker and one maintenance thread even after rerunning this cell.
重跑 Cell 仍僅保留一個 worker 和一個清理 thread。
'''
if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()
if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

'''
Close only the previous ngrok tunnel; preserve the running API and its jobs.
只關閉舊的 ngrok 通道，保留正在運作的 API 與所有生成工作。
'''
old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)
NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(Path('/kaggle/working/ngrok'))
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

'''
Always request the auth token again when this cell is rerun.
每次重跑均重新要求輸入 ngrok Authtoken，不顯示內容。
'''
ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')
result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')
del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

'''
Reuse the same API server when this cell is executed again.
重跑本格時沿用同一個 API Server，不中斷生成工作或清空佇列。
'''
server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()
if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:

    '''
    Update the handler on rerun without restarting an active GPU job.
    重跑後更新 API handler，但不重新啟動正在執行的生成工作。
    '''
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')
with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)
NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)
if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])
print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')


## 7. Web API 使用說明

In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request

'''
Request the API schema directly from the local API server.
直接從本機 API Server 查詢支援的參數與模式。
'''
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)
print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本地 requests 範例：06_Z-Image-Turbo_圖片生成_Kaggle_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')